In [1]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "input.txt"
)

('input.txt', <http.client.HTTPMessage at 0x7d4bb00e96d0>)

## Setup & Data Loading

Notes from KV cache implementation:
- During inference (after prefill), only one new token arrives each step
- Q is computed from only the new token (size 1)
- K and V can be concatenated onto whatever you already cached
- Q @ K^T still works if Q has shape (B, 1, hs) and K has shape (B, T, hs) → result is (B, 1, T)
- We don't need the causal mask during decode because we are only considering the most recent token

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import time
from dataclasses import dataclass, field
from typing import List, Dict, Tuple

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss, _ = model(X, Y)  # unpack 3 return values now
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## Bigram Model
At 210K params, your GPT is already tiny. A "smaller GPT" would be absurd. Instead, use a **bigram model**: a simple lookup table that predicts the next token based only on the current token.

**Why bigram?** It's essentially free — a single table lookup vs. a full transformer forward pass. The draft quality will be mediocre (bigrams can't capture long-range dependencies), but that's fine. Even a 30% acceptance rate means you sometimes get 2-3 tokens per target forward pass instead of 1.

**Why Laplace smoothing?** Without it, some bigram entries are zero. If the draft assigns probability 0 to a token that the target model wants, the rejection sampling math breaks (division by zero).

In [3]:
class BigramDraftModel:
    """
    Draft model for speculative decoding.
    Predicts P(next_token | current_token) from training data statistics.
    """
    def __init__(self, train_data, vocab_size, device):
        # Count bigram frequencies: how often does token B follow token A?
        counts = torch.zeros(vocab_size, vocab_size, device=device)
        for i in range(len(train_data) - 1):
            counts[train_data[i], train_data[i + 1]] += 1

        # Convert counts to probabilities (add smoothing to avoid zeros)
        counts += 1  # Laplace smoothing
        self.probs = counts / counts.sum(dim=1, keepdim=True)  # (vocab_size, vocab_size)

    def get_probs(self, token_id):
        """Return P(next | token_id) as a (vocab_size,) distribution."""
        return self.probs[token_id]

    def sample(self, token_id):
        """Sample one token given the current token."""
        probs = self.get_probs(token_id)
        return torch.multinomial(probs, num_samples=1).item(), probs

## Speculative Decode Loop

In [4]:
def speculative_generate(target_model, draft_model, prompt_tokens, max_new_tokens, K=4):
    """
    Generate tokens using speculative decoding.

    Args:
        target_model:  your GPTLanguageModel (the "real" model)
        draft_model:   BigramDraftModel (the cheap guesser)
        prompt_tokens: list of ints — the encoded prompt
        max_new_tokens: how many tokens to generate
        K:             number of draft tokens per speculation step (gamma in the paper)

    Returns:
        generated_tokens: list of ints
    """
    generated = []
    # You'll need to manage the target model's KV cache here.
    # Start by prefilling the prompt through the target model.

    while len(generated) < max_new_tokens:
        # 1. DRAFT: generate K candidate tokens using the cheap model
        
        
        # 2. VERIFY: run all K candidates through the target model in ONE forward pass
        # 3. ACCEPT/REJECT: compare draft vs target probabilities, accept or resample

        # ... (see Hints 3-5 for details)
        pass

    return generated[:max_new_tokens]

## Draft Phase: Generating K Tokens

The draft phase is simple: autoregressively sample K tokens from the bigram model, storing the draft probabilities for each.

In [5]:
def draft_tokens(draft_model, current_token, K):
    """
    Generate K speculative tokens from the draft model.

    Returns:
        candidates:  list of K token ids
        draft_probs: list of K probability distributions (each is (vocab_size,))
    """
    candidates = []
    draft_probs = []
    tok = current_token

    for _ in range(K):
        next_tok, probs = draft_model.sample(tok)
        candidates.append(next_tok)
        draft_probs.append(probs)
        tok = next_tok

    return candidates, draft_probs

## Verify Candidates


In [6]:
def verify_candidates(target_model, current_token, candidates, past_kvs):
    """
    Run the target model on [current_token] + candidates in one forward pass.

    Returns:
        target_probs: list of K+1 probability distributions
        new_kvs:      updated KV cache
    """

    all_tokens = [current_token] + candidates
    input_ids = torch.tensor([all_tokens], dtype=torch.long, device=device)

    cache_len = 0
    if past_kvs is not None:
        cache_len = past_kvs[0][0][0].shape[1]
    
    positions = torch.arange(cache_len, cache_len + len(all_tokens), device=device).unsqueeze(0)

    logits, _, new_kvs = target_model(input_ids, positions, past_kvs)

    # Convert to probabilities
    target_probs = []
    for i in range(len(all_tokens)):
        probs = F.softmax(logits[0, i, :], dim=-1)
        target_probs.append(probs)

    return target_probs, new_kvs    


## Accept/Reject Phase - Rejection Sampling

This is the mathematically precise part. For each draft token, you decide whether to accept it based on how well the draft model's prediction matches the target model's prediction.

In [7]:
def accept_reject(candidates, draft_probs, target_probs):
    """
    Apply rejection sampling to decide which draft tokens to accept.

    Args:
        candidates:   list of K draft token ids
        draft_probs:  list of K draft probability distributions
        target_probs: list of K+1 target probability distributions
                      (target_probs[i] is the target's distribution for position i)

    Returns:
        accepted_tokens: list of accepted tokens (1 to K+1 tokens)
    """
    accepted = []

    for i in range(len(candidates)):
        token = candidates[i]
        q = draft_probs[i][token]    # draft model's probability for this token
        p = target_probs[i][token]   # target model's probability for this token

        # Accept with probability min(1, p/q)
        if torch.rand(1, device=p.device).item() < (p / q).clamp(max=1.0).item():
            accepted.append(token)
        else:
            # Rejected! Resample from the adjusted distribution
            # The adjusted distribution ensures we match the target exactly
            adjusted = torch.clamp(target_probs[i] - draft_probs[i], min=0)
            adjusted = adjusted / adjusted.sum()
            resampled = torch.multinomial(adjusted, num_samples=1).item()
            accepted.append(resampled)
            return accepted  # Stop here — don't check further candidates

    # All K candidates accepted! Sample one bonus token from the target
    bonus = torch.multinomial(target_probs[len(candidates)], num_samples=1).item()
    accepted.append(bonus)
    return accepted

## Hint 6: KV Cache Management

The tricky part: after accept/reject, you need to **trim the KV cache** to match only the accepted tokens. The verification forward pass cached KV entries for all K+1 tokens, but if you only accepted 2 of them, the cache entries for positions 3, 4, 5 are invalid.

In [8]:
def trim_kv_cache(new_kvs, num_accepted, cache_len_before_verify):
    """
    Trim the KV cache to only include accepted tokens.

    After verification, the cache has entries for all K+1 speculative tokens.
    We need to keep only the first `num_accepted` new entries.

    Args:
        new_kvs:     KV cache from the verification forward pass
        num_accepted: how many tokens were accepted
        cache_len_before_verify: KV cache length before the verify call
    """
    keep = cache_len_before_verify + num_accepted
    trimmed = []
    for layer_kv in new_kvs:
        layer_trimmed = []
        for (k, v) in layer_kv:
            layer_trimmed.append((k[:, :keep, :], v[:, :keep, :]))
        trimmed.append(layer_trimmed)
    return trimmed

## The Global Block Cache

A prompt of 12 tokens becomes 3 blocks:

```text
Block 0: tokens[0:4]   → KV for positions 0, 1, 2, 3
Block 1: tokens[4:8]   → KV for positions 4, 5, 6, 7
Block 2: tokens[8:12]  → KV for positions 8, 9, 10, 11

```
Each block stores a fixed-size KV chunk: (1, BLOCK_SIZE, head_size) per (layer, head). Only full blocks (exactly BLOCK_SIZE tokens) are eligible for caching. The trailing partial block is never cached — it changes with every new decode token.

Question to ask yourself: Why can't you cache partial blocks?

Answer: because the block's content isn't finalized. During decode, new tokens append to the last block until it fills up. Only then is its token content fixed and its hash meaningful.

You need a global cache that lives outside any individual request — a shared pool that multiple requests can read from.

In [9]:
import hashlib

NONE_HASH = b'\x00' * 16  # sentinel for the first block (no parent)

def hash_block_tokens(parent_hash, token_ids):
    """Compute a chained content hash for a KV block."""
    data = (parent_hash, tuple(token_ids))
    return hashlib.md5(str(data).encode()).digest()

@dataclass
class CachedBlock:
    """A cached KV block with its content hash."""
    block_hash: bytes
    token_ids: tuple                    # the tokens this block covers
    kv_data: Dict[Tuple[int, int], Tuple[torch.Tensor, torch.Tensor]]
    # kv_data[(layer, head)] = (k, v), each (1, BLOCK_SIZE, head_size)
    last_access_step: int = 0          # for LRU eviction

class BlockCache:
    def __init__(self, max_blocks=64):
        self.max_blocks = max_blocks
        self.cache: Dict[bytes, CachedBlock] = {}  # hash → CachedBlock
        self.current_step = 0

    def lookup(self, block_hash) -> CachedBlock | None:
        """Look up a block by its content hash."""
        block = self.cache.get(block_hash)
        if block is not None:
            block.last_access_step = self.current_step  # touch for LRU
        return block

    def insert(self, block_hash, token_ids, kv_data):
        """Insert a completed block into the cache."""
        if len(self.cache) >= self.max_blocks:
            self._evict_lru()
            
        self.cache[block_hash] = CachedBlock(
            block_hash=block_hash,
            token_ids=token_ids,
            kv_data=kv_data,
        )

    def _evict_lru(self):
        """Evict the least-recently-used block."""
        oldest = min(self.cache.values(), key=lambda b: b.last_access_step)
        del self.cache[oldest.block_hash]

In [10]:
class KVBlockPool:
    """
    Pre-allocated GPU memory pool for KV cache blocks.
    
    Physical layout: one big tensor per (layer, head, k/v).
    Shape: (num_physical_blocks, block_size, head_size)
    
    Block i occupies pool[i, :, :] — a fixed-size (block_size, head_size) slab.
    """

    def __init__(self, num_blocks, block_size, n_layer, n_head, head_size, device):
        self.num_blocks = num_blocks
        self.block_size = block_size
        self.k_pool = {}
        self.v_pool = {}

        for layer in range(n_layer):
            for head in range(n_head):
                self.k_pool[(layer, head)] = torch.zeros(
                    num_blocks, block_size, head_size, device=device
                )

                self.v_pool[(layer, head)] = torch.zeros(
                    num_blocks, block_size, head_size, device=device
                )


## Finding Cache Hits During Admission

When a new request arrives, the scheduler needs to figure out how many of its prompt tokens are already cached. This is done by computing block hashes from the prompt and checking each one against the BlockCache:

In [11]:
def find_cached_prefix(block_cache: BlockCache, prompt_tokens, block_size):
    """
        Walk the prompt left-to-right in block-sized chunks.
        Return the number of tokens that are fully cached
    """

    num_cached = 0
    parent_hash = NONE_HASH

    for start in range(0, len(prompt_tokens), block_size):
        end = start + block_size
        if end > len(prompt_tokens): break

        chunk = prompt_tokens[start:end]
        chunk_hash = hash_block_tokens(parent_hash, chunk)

        cached_block = block_cache.lookup(chunk_hash)

        if cached_block is None: break

        num_cached += block_size
        parent_hash = chunk_hash
    
    return num_cached

## Request Dataclass

Each in-flight generation carries its own state:
- `prompt_tokens` — the initial context
- `max_new_tokens` — individual stopping condition (request 0 may want 20, request 1 may want 100)
- `generated_tokens` — accumulates one token per scheduler step
- `status` — lets the scheduler know whether to batch this request
- `kv_cache` — **per-request** KV cache, keyed by `(layer_idx, head_idx)`

If request 0 finishes after 20 tokens but request 1 needs 100, request 0 is evicted from the batch (its row disappears), and request 1 continues generating.

In [12]:
@dataclass
class Request:
    """Each in-flight generation carries its own state and KV cache."""
    id: int
    prompt_tokens: List[int]          # the original encoded prompt
    max_new_tokens: int               # how many tokens this request wants
    generated_tokens: List[int] = field(default_factory=list)
    status: str = "waiting"           # "waiting" -> "prefilling" -> "active" -> "done"
    prefill_cursor: int = 0
    _committed_blocks: int = 0

    block_table: List[int] = field(default_factory=list)
    num_filled_slots: int = 0

    @property
    def tokens_so_far(self) -> List[int]:
        """Full sequence: prompt + everything generated."""
        return self.prompt_tokens + self.generated_tokens

    @property
    def num_tokens_in_cache(self):
        return self.num_filled_slots        

    @property
    def num_generated(self) -> int:
        return len(self.generated_tokens)

    @property
    def is_done(self) -> bool:
        return self.num_generated >= self.max_new_tokens
    
    @property
    def is_fully_prefilled(self) -> bool:
        return self.prefill_cursor == len(self.prompt_tokens)

    def clear_cache(self, block_allocator):
        block_allocator.free_blocks_for_request(self.block_table)
        self.block_table = []
        self.num_filled_slots = 0

In [13]:
def write_kv_to_pool(pool, block_table, block_size, start_pos, k_new, v_new, layer, head):
    """
    Write new KV data into the physical pool using the block table.
    
    Args:
        pool:        KVBlockPool
        block_table: list of physical block indices for this request
        block_size:  tokens per block
        start_pos:   logical position of the first new token
        k_new:       (1, T_new, head_size) — new key data
        v_new:       (1, T_new, head_size) — new value data
    """

    T_new = k_new.shape[1]

    for t in range(T_new):
        logical_pos = start_pos + t
        block_idx = logical_pos // block_size
        slot_idx = logical_pos % block_size
        phys_block = block_table[block_idx]

        pool.k_pool[(layer, head)][phys_block, slot_idx, :] = k_new[0, t, :]
        pool.v_pool[(layer, head)][phys_block, slot_idx, :] = v_new[0, t, :]

def maybe_allocate_block(request, block_allocator, block_size):
    """Allocate a new physical block if the current one is full."""
    if request.num_filled_slots % block_size == 0:
        new_block = block_allocator.allocate_one()
        request.block_table.append(new_block)


def gather_kv_from_pool(pool, block_table, block_size, num_filled, layer, head):
    """
    Gather a request's KV cache from the physical pool into a contiguous tensor.
    
    Returns:
        k: (1, num_filled, head_size)
        v: (1, num_filled, head_size)
    """

    if num_filled == 0:
        hs = pool.k_pool[(layer, head)].shape[-1]
        device = pool.k_pool[(layer, head)].device
        return (
            torch.empty(1, 0, hs, device=device),
            torch.empty(1, 0, hs, device=device),
        )        

    num_full_blocks = num_filled // block_size
    trailing_slots = num_filled % block_size

    k_parts = []
    v_parts = []

    for i in range(num_full_blocks):
        phys = block_table[i]

        k_parts.append(pool.k_pool[(layer, head)][phys])
        v_parts.append(pool.v_pool[(layer, head)][phys])

    
    if trailing_slots > 0:
        phys = block_table[num_full_blocks]
        k_parts.append(pool.k_pool[(layer, head)][phys, :trailing_slots])
        v_parts.append(pool.v_pool[(layer, head)][phys, :trailing_slots])

    k = torch.cat(k_parts, dim=0).unsqueeze(0)
    v = torch.cat(v_parts, dim=0).unsqueeze(0)

    return k, v


In [14]:
class BlockAllocator:
    def __init__(self, num_blocks):
        self.num_blocks = num_blocks
        self.free_blocks = list(range(num_blocks))
        self.block_size = block_size

    def allocate_one(self):
        if not self.free_blocks:
            return RuntimeError("Block pool exhausted")
        
        return self.free_blocks.pop()
    
    def allocate_n(self, n):
        if not self.free_blocks or len(self.free_blocks) < n:
            return RuntimeError("Block pool exhausted")

        return [self.free_blocks.pop() for _ in range(n)]  
    
    def free_blocks_for_request(self, block_table):
        self.free_blocks.extend(block_table)
    
    @property
    def num_free(self):
        return len(self.free_blocks)


In [15]:
class Scheduler:
    def __init__(self, policy="fcfs", max_batch_size=4, token_budget=16, max_kv_tokens=22, block_size=4):
        self.policy = policy
        self.max_batch_size = max_batch_size
        self.token_budget = token_budget
        self.max_kv_tokens = max_kv_tokens
        self.block_size = block_size
        self.block_cache = BlockCache()
        self.block_allocator = None
        self.current_compute_tokens = 0

        self.waiting = []
        self.prefilling = []
        self.active = []
        self.preempted = []

    def promote(self, req):
        self.prefilling.remove(req)
        req.status = "active"
        self.active.append(req)
    
    def complete(self, req):
        if req in self.active:
            
            self.active.remove(req)
        req.status = "done"
        self.block_allocator.free_blocks_for_request(req.block_table)

    def _sort_key(self, req):
        if self.policy == "fcfs":
            return (0, req.arrival_time)
        elif self.policy == "priority":
            return (req.priority, req.arrival_time)
    
    def add_request(self, req):
        key = self._sort_key(req)
        heapq.heappush(self.waiting, (*key, req.id, req))
    
    def is_done(self):
        return not (self.waiting or self.prefilling or self.active)
    
    def _maybe_admit(self, step):
        if not self.waiting: return

        candidate = self.waiting[0]
        prompt_len = len(candidate.prompt_tokens)

        blocks_needed = (prompt_len + self.block_size - 1) // self.block_size

        if self.block_allocator.num_free < blocks_needed:
            return
        
        needed_compute = min(prompt_len, self.token_budget)

        if self.current_compute_tokens + needed_compute > self.token_budget:
            return

        heapq.heappop(self.waiting)
        candidate.status = "prefilling"

        candidate.block_table = self.block_allocator.allocate_n(blocks_needed)

        candidate.num_filled_slots = 0

        self.prefilling.append(candidate)

    
    def _maybe_preempt(self):
        kv_used = sum(len(req.prompt_tokens) + req.num_generated for req in self.active + self.prefilling)

        while self.active and kv_used > self.max_kv_tokens:
            victim = max(self.active, key=lambda r: (r.priority, -r.arrival_time))
            self.active.remove(victim)
            victim.clear_cache(self.block_allocator)
            victim.prefill_cursor = 0
            victim.generated_tokens = []
            victim.status = "waiting"
            self.preempted.append(victim)

            key = self._sort_key(victim)
            heapq.heappush(self.waiting, (*key, victim.id, victim))
            kv_used = sum(len(req.prompt_tokens) + req.num_generated for req in self.active + self.prefilling)

    def schedule(self, step: int):
        """
        Returns:
            prefill_req:  Request | None  — one request getting a prefill chunk (or None)
            decode_reqs:  List[Request]   — all requests currently being decoded (active)

        """
        self.block_cache.current_step = step

        self._maybe_admit(step)       # promote waiting → prefilling if memory allows
        self._maybe_preempt()         # evict if over memory budget

        prefill_req = self.prefilling[0] if self.prefilling else None
        decode_reqs = list(self.active)

        return prefill_req, decode_reqs


## Hint 2: Stateless Head — KV Cache Moved Outside the Model

**Before:** `Head` owned `self.key_cache` and `self.value_cache` — one monolithic `(B, T, hs)` tensor.
This breaks when different requests have different sequence lengths.

**After:** `Head` is stateless. The cache is:
1. Passed **into** `forward()` as `past_k, past_v`
2. Returned **out of** `forward()` as updated `(new_k, new_v)`
3. **Stored on the `Request` object**, keyed by `(layer_idx, head_idx)`

This threads through: `Head` → `MultiHeadAttention` → `Block` → `GPTLanguageModel`.

Also changed `nn.Sequential` → `nn.ModuleList` for `self.blocks` so we can pass
per-block cache into each block individually.

In [16]:
class Head(nn.Module):
    """One head of self-attention — now STATELESS (no internal cache)."""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_k=None, past_v=None, attn_mask=None):
        """
        Args:
            x:      (B, T, C)       input embeddings
            past_k: (B, T_past, hs) cached keys, or None
            past_v: (B, T_past, hs) cached values, or None
        Returns:
            out:   (B, T, hs)           attention output
            new_k: (B, T_past+T, hs)    updated key cache   (None during training)
            new_v: (B, T_past+T, hs)    updated value cache  (None during training)
        """
        B, T, C = x.shape
        k = self.key(x)    # (B, T, hs)
        q = self.query(x)  # (B, T, hs)
        v = self.value(x)  # (B, T, hs)

        if not self.training:
            if past_k is not None:
                k = torch.cat([past_k, k], dim=1)
                v = torch.cat([past_v, v], dim=1)

            T_full = k.shape[1]

            wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5  # (B, T, T_full)

            causal_mask = torch.ones(T, T_full, device=x.device, dtype=torch.bool)

            if T > 1:
                new_token_mask = self.tril[:T, :T]
                causal_mask[:, -T:] = new_token_mask

            causal_mask = causal_mask.unsqueeze(0).expand(B, -1,- 1)
        
            if attn_mask is not None:
                new_valid = torch.ones(B, 1, T, device=x.device, dtype=torch.bool)
                full_pad_mask = torch.cat([attn_mask, new_valid], dim=-1)
                causal_mask = causal_mask & full_pad_mask

            wei = wei.masked_fill(~causal_mask, float("-inf"))
            wei = F.softmax(wei, dim=-1)
            wei = self.dropout(wei)
            out = wei @ v

            return out, k, v

        else:
            wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
            wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
            wei = F.softmax(wei, dim=-1)
            wei = self.dropout(wei)
            out = wei @ v
            return out, None, None            

class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel."""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, attn_mask=None):
        """
        Args:
            x:       (B, T, C)
            past_kv: list of (past_k, past_v) per head, or None
        Returns:
            out:    (B, T, n_embd)
            new_kv: list of (new_k, new_v) per head
        """
        if past_kv is None:
            past_kv = [(None, None)] * len(self.heads)

        outputs, new_kvs = [], []
        for i, h in enumerate(self.heads):
            pk, pv = past_kv[i]
            out, nk, nv = h(x, pk, pv, attn_mask=attn_mask)
            outputs.append(out)
            new_kvs.append((nk, nv))

        out = torch.cat(outputs, dim=-1)
        out = self.dropout(self.proj(out))
        return out, new_kvs


class FeedFoward(nn.Module):
    """A simple linear layer followed by a non-linearity."""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block: communication followed by computation."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x, past_kv=None, attn_mask=None):
        """
        Returns:
            x:      (B, T, n_embd)
            new_kv: list of (new_k, new_v) per head in this block
        """
        sa_out, new_kv = self.sa(self.ln1(x), past_kv, attn_mask=attn_mask)
        x = x + sa_out
        x = x + self.ffwd(self.ln2(x))
        return x, new_kv


class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # ModuleList instead of Sequential so we can pass per-block cache
        self.blocks = nn.ModuleList([Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, pos=None, past_kvs=None, attn_mask=None):
        """
        Args:
            idx:      (B, T) token indices
            targets:  (B, T) target indices, or None
            pos:      (B, T) explicit position indices, or None (uses arange)
            past_kvs: list-of-lists cache structure, or None
                      past_kvs[layer][head] = (key_tensor, value_tensor)
        Returns:
            logits:   (B, T, vocab_size)
            loss:     scalar or None
            new_kvs:  updated cache with same structure as past_kvs
        """
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)  # (B, T, C)

        if pos is None:
            pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T, C)
        else:
            pos_emb = self.position_embedding_table(pos)  # (B, T, C)

        x = tok_emb + pos_emb  # (B, T, C)

        # Thread cache through each block
        if past_kvs is None:
            past_kvs = [None] * len(self.blocks)

        new_kvs = []
        for i, block in enumerate(self.blocks):
            x, block_kv = block(x, past_kvs[i], attn_mask=attn_mask)
            new_kvs.append(block_kv)

        x = self.ln_f(x)          # (B, T, C)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss, new_kvs

    def generate(self, idx, max_new_tokens):
        """Original generate (no cache, full recompute) for reference."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## Training

Training is **unchanged** — during `model.train()`, every `Head` takes the training branch
and returns `(out, None, None)` for the cache. The cache is simply discarded via `_`.

In [17]:
model = GPTLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss, _ = model(xb, yb)  # _ discards the cache during training
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Quick sanity check with the original no-cache generate
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=200)[0].tolist()))

0.209729 M parameters
step 0: train loss 4.1959, val loss 4.1962
step 100: train loss 2.6229, val loss 2.6166
step 200: train loss 2.4555, val loss 2.4488
step 300: train loss 2.3810, val loss 2.3928
step 400: train loss 2.3202, val loss 2.3223
step 500: train loss 2.2364, val loss 2.2541
step 600: train loss 2.1812, val loss 2.2234
step 700: train loss 2.1326, val loss 2.1583
step 800: train loss 2.0932, val loss 2.1352
step 900: train loss 2.0499, val loss 2.0987
step 1000: train loss 2.0349, val loss 2.0819
step 1100: train loss 1.9994, val loss 2.0706
step 1200: train loss 1.9857, val loss 2.0726
step 1300: train loss 1.9638, val loss 2.0401
step 1400: train loss 1.9386, val loss 2.0296
step 1500: train loss 1.9028, val loss 1.9947
step 1600: train loss 1.8821, val loss 1.9914
step 1700: train loss 1.8785, val loss 1.9813
step 1800: train loss 1.8746, val loss 1.9878
step 1900: train loss 1.8451, val loss 1.9599
step 2000: train loss 1.8294, val loss 1.9525
step 2100: train loss 1.

## The Scheduler Loop

Now that each request owns its own KV cache, the next step is the **scheduler loop**:

```
while there are active requests OR the waiting queue is non-empty:
    1. Check the waiting queue — can any new requests join the batch?
    2. Build the input tensor from ALL active requests (each contributes 1 token)
    3. Forward pass → get logits for all active requests at once
    4. Sample next token for each request
    5. Check: did any request hit max_new_tokens? → remove it, emit result
    6. Go to 1
```

The key challenge will be **padding the KV caches** to a common T dimension
when batching multiple requests (since they have different sequence lengths).
You'll need to `torch.cat` along dim=0 after padding along dim=1.

After un-batching the results, scatter the updated caches back to each request.

In [18]:
def assemble_paged_cache(requests, pool, block_size):
    """
    Gather per-request KV from the paged pool into batched tensors.
    Same interface as assemble_batch_cache — returns left-padded batched cache.
    """

    B = len(requests)

    lengths = [req.num_filled_slots for req in requests]
    max_t = max(lengths) if lengths else 0
    pad_lengths = [max_t - t for t in lengths]

    attn_mask = torch.zeros(B, 1, max_t, device=device, dtype=torch.bool)
    for i, pad in enumerate(pad_lengths):
        attn_mask[i, 0, pad:] = True
    
    past_kvs = []

    for layer_idx in range(n_layer):
        block_kv = []
        for head_idx in range(n_head):
            keys, values = [], []
            for i, req in enumerate(requests):
                k, v = gather_kv_from_pool(
                    pool, req.block_table, block_size, 
                    req.num_filled_slots, layer_idx, head_idx
                )

                if pad_lengths[i] > 0:
                    hs = k.shape[2]
                    pad_tensor = torch.zeros(1, pad_lengths[i], hs, device=device)
                    k = torch.cat([pad_tensor, k], dim=1)
                    v = torch.cat([pad_tensor, v], dim=1)
                
                keys.append(k)
                values.append(v)
            
            block_kv.append((torch.cat(keys, dim=0), torch.cat(values, dim=0)))

        past_kvs.append(block_kv)
    
    return past_kvs, attn_mask, pad_lengths

def disassemble_paged_cache(requests, new_kvs, pad_lengths, pool, block_size):
    """
    Scatter new KV data from model output back into the paged pool.
    Each request gets 1 new KV entry (decode token).
    """

    for layer_idx, block_kv in enumerate(new_kvs):
        for head_idx, (batched_k, batched_v) in enumerate(block_kv):
            for i, req in enumerate(requests):
                pad = pad_lengths[i]

                k_new = batched_k[i:i+1, -1:, :]
                v_new = batched_v[i:i+1, -1:, :]

                write_kv_to_pool(pool, req.block_table, block_size,
                req.num_filled_slots, k_new, v_new, layer_idx, head_idx)

    for req in requests:
        req.num_filled_slots += 1

def disassemble_paged_fused(all_reqs, new_kvs, num_new_per_req, pool, block_size):
    """Like disassemble_paged_cache but handles variable new tokens per row."""
    for layer_idx, block_kv in enumerate(new_kvs):
        for head_idx, (batched_k, batched_v) in enumerate(block_kv):
            for i, req in enumerate(all_reqs):
                t_new = num_new_per_req[i]
                k_new = batched_k[i:i+1, -t_new:, :]
                v_new = batched_v[i:i+1, -t_new:, :]
                
                write_kv_to_pool(
                    pool, req.block_table, block_size,
                    req.num_filled_slots,
                    k_new, v_new, layer_idx, head_idx
                )
    
    for i, req in enumerate(all_reqs):
        req.num_filled_slots += num_new_per_req[i]        

def assemble_fused_batch(decode_reqs: List[Request], prefill_req, chunk_size, pool, block_size):
    """
    Build a single (B, T_max) input tensor + batched cache for the fused forward pass.

    Args:
        decode_reqs:  list of active Request objects (each contributes 1 token)
        prefill_req:  the request being prefilled (contributes chunk_size tokens), or None
        chunk_size:   number of prefill tokens this step

    Returns:
        batch_tokens:   (B, T_max) input tensor
        batch_positions: (B, T_max) position indices
        past_kvs:       batched cache [layer][head] = (B, T_max_cache, hs)
        attn_mask:      (B, 1, T_max_cache) bool mask for cached positions
        pad_info:       dict with per-row metadata for disassembly
    """

    num_new_tokens = []
    all_reqs = []

    for req in decode_reqs:
        all_reqs.append(req)
        num_new_tokens.append(1)
    
    if prefill_req:
        all_reqs.append(prefill_req)
        num_new_tokens.append(chunk_size)

    B = len(all_reqs)
    T_max = max(num_new_tokens)

    batch_tokens = []
    batch_positions = []

    for req in decode_reqs:
        pos_val = len(req.tokens_so_far) - 1
        row = [0] * (T_max - 1) + [pos_val]
        batch_positions.append(row)

        token_row = [0] * (T_max - 1) + [req.tokens_so_far[-1]]
        batch_tokens.append(token_row)
    
    if prefill_req:
        cursor = prefill_req.prefill_cursor

        chunk_positions = list(range(cursor, cursor + chunk_size))

        padding = [0] * (T_max - chunk_size)

        batch_positions.append(padding + chunk_positions)

        chunk = prefill_req.prompt_tokens[cursor: cursor + chunk_size]
        pad = [0] * (T_max - chunk_size)

        batch_tokens.append(pad + chunk)

    batch_positions = torch.tensor(batch_positions, device=device)        
    batch_tokens = torch.tensor(batch_tokens, dtype=torch.long, device=device)  
    # Assemble KV cache 
        
    past_kvs, attn_mask, pad_lengths = assemble_paged_cache(all_reqs, pool, block_size)    
    return batch_tokens, batch_positions, past_kvs, attn_mask, pad_lengths

def commit_completed_blocks(request: Request, block_cache: BlockCache, block_size, pool):
    """
    After a prefill step, check if any new full blocks were completed.
    If so, insert them into the global cache.
    """

    total_tokens = len(request.prompt_tokens) + request.num_generated
    num_full_blocks = request.prefill_cursor // block_size    

    parent_hash = NONE_HASH

    for block_idx in range(num_full_blocks):
        start = block_idx * block_size
        end = start + block_size

        chunk = request.prompt_tokens[start:end]
        block_hash = hash_block_tokens(parent_hash, chunk)

        if block_idx >= request._committed_blocks:
            kv_data = {}
            phys_block = request.block_table[block_idx]
            for layer in range(n_layer):
                for head in range(n_head):
                    kv_data[(layer, head)] = (
                        pool.k_pool[(layer, head)][phys_block].unsqueeze(0).clone(),
                        pool.v_pool[(layer, head)][phys_block].unsqueeze(0).clone(),
                    )

            block_cache.insert(block_hash, tuple(chunk), kv_data)

        parent_hash = block_hash
    
    request._committed_blocks = num_full_blocks

In [19]:
@torch.no_grad()
def speculative_generate(target_model, draft_model, prompt_tokens, max_new_tokens, K=4):
    target_model.eval()
    generated = []

    # 1. Prefill: run the full prompt through the target model
    input_ids = torch.tensor([prompt_tokens], dtype=torch.long, device=device)
    positions = torch.arange(len(prompt_tokens), device=device).unsqueeze(0)
    logits, _, past_kvs = target_model(input_ids, pos=positions)

    # Sample the first token from the prefill output
    probs = F.softmax(logits[0, -1, :], dim=-1)
    current_token = torch.multinomial(probs, num_samples=1).item()
    generated.append(current_token)

    # 2. Speculative decode loop
    while len(generated) < max_new_tokens:
        cache_len = past_kvs[0][0][0].shape[1]  # current KV cache length

        # How many tokens to speculate (don't overshoot max_new_tokens)
        k = min(K, max_new_tokens - len(generated))

        # DRAFT
        candidates, draft_probs = draft_tokens(draft_model, current_token, k)

        # VERIFY
        target_probs, new_kvs = verify_candidates(
            target_model, current_token, candidates, past_kvs
        )

        # ACCEPT/REJECT
        accepted = accept_reject(candidates, draft_probs, target_probs)

        # TRIM KV CACHE (keep only accepted tokens)
        past_kvs = trim_kv_cache(new_kvs, len(accepted), cache_len)

        # Update state
        generated.extend(accepted)
        current_token = accepted[-1]

    return generated[:max_new_tokens]


## Hint 8: Measuring the Speedup

The metric that matters is **acceptance rate** — what fraction of draft tokens get accepted on average.

In [20]:
def benchmark_speculative(target_model, draft_model, prompts, max_new_tokens=50, K=4):
    """Compare speculative vs standard decoding."""

    # Count target model forward passes
    target_calls_standard = 0
    target_calls_speculative = 0
    total_tokens = 0
    total_accepted = 0
    total_drafted = 0

    # ... run both approaches, count forward passes ...

    print(f"Standard decoding: {target_calls_standard} target forward passes")
    print(f"Speculative decoding: {target_calls_speculative} target forward passes")
    print(f"Acceptance rate: {total_accepted / total_drafted:.1%}")
    print(f"Tokens per target call: {total_tokens / target_calls_speculative:.2f}")
    print(f"Speedup: {target_calls_standard / target_calls_speculative:.2f}x")

In [21]:
# ─── DRAFT MODEL ───
class BigramDraftModel:
    """Bigram lookup table as a cheap draft model."""
    def __init__(self, train_data, vocab_size, device):
        counts = torch.zeros(vocab_size, vocab_size, device=device)
        for i in range(len(train_data) - 1):
            counts[train_data[i], train_data[i + 1]] += 1
        counts += 1  # Laplace smoothing
        self.probs = counts / counts.sum(dim=1, keepdim=True)
    def get_probs(self, token_id):
        return self.probs[token_id]
    def sample(self, token_id):
        probs = self.get_probs(token_id)
        return torch.multinomial(probs, num_samples=1).item(), probs
draft_model = BigramDraftModel(train_data, vocab_size, device)

In [22]:
# ─── SPECULATIVE DECODING FUNCTIONS ───
def draft_tokens(draft_model, current_token, K):
    """Generate K candidates from the bigram draft model."""
    candidates = []
    draft_probs = []
    tok = current_token
    for _ in range(K):
        next_tok, probs = draft_model.sample(tok)
        candidates.append(next_tok)
        draft_probs.append(probs)
        tok = next_tok
    return candidates, draft_probs

def verify_candidates(target_model, current_token, candidates, past_kvs):
    """Run [current_token, c0, ..., c_{K-1}] through target in one forward pass."""
    all_tokens = [current_token] + candidates
    input_ids = torch.tensor([all_tokens], dtype=torch.long, device=device)
    cache_len = 0
    if past_kvs is not None and past_kvs[0] is not None:
        cache_len = past_kvs[0][0][0].shape[1]
    positions = torch.arange(cache_len, cache_len + len(all_tokens), device=device).unsqueeze(0)
    logits, _, new_kvs = target_model(input_ids, pos=positions, past_kvs=past_kvs)
    target_probs = [F.softmax(logits[0, i, :], dim=-1) for i in range(len(all_tokens))]
    return target_probs, new_kvs

def accept_reject(candidates, draft_probs, target_probs):
    """Rejection sampling: accept draft tokens or resample from adjusted distribution."""
    accepted = []
    for i in range(len(candidates)):
        token = candidates[i]
        q = draft_probs[i][token]
        p = target_probs[i][token]
        if torch.rand(1, device=p.device).item() < (p / q).clamp(max=1.0).item():
            accepted.append(token)
        else:
            adjusted = torch.clamp(target_probs[i] - draft_probs[i], min=0)
            adj_sum = adjusted.sum()
            if adj_sum > 0:
                adjusted = adjusted / adj_sum
            else:
                adjusted = target_probs[i]  # fallback
            resampled = torch.multinomial(adjusted, num_samples=1).item()
            accepted.append(resampled)
            return accepted  # stop on first rejection
    # All accepted — sample bonus token from target
    bonus = torch.multinomial(target_probs[len(candidates)], num_samples=1).item()
    accepted.append(bonus)
    return accepted
def trim_kv_cache(new_kvs, num_to_keep):
    """Trim KV cache to only include entries for accepted tokens."""
    trimmed = []
    for layer_kv in new_kvs:
        layer_trimmed = []
        for (k, v) in layer_kv:
            layer_trimmed.append((k[:, :num_to_keep, :], v[:, :num_to_keep, :]))
        trimmed.append(layer_trimmed)
    return trimmed


In [23]:
# ─── GENERATE FUNCTIONS ───
@torch.no_grad()
def speculative_generate(target_model, draft_model, prompt_tokens, max_new_tokens, K=4):
    """Generate with speculative decoding (draft + verify)."""
    target_model.eval()
    generated = []
    total_target_calls = 0
    # Prefill
    input_ids = torch.tensor([prompt_tokens], dtype=torch.long, device=device)
    positions = torch.arange(len(prompt_tokens), device=device).unsqueeze(0)
    logits, _, past_kvs = target_model(input_ids, pos=positions)
    total_target_calls += 1
    probs = F.softmax(logits[0, -1, :], dim=-1)
    current_token = torch.multinomial(probs, num_samples=1).item()
    generated.append(current_token)
    while len(generated) < max_new_tokens:
        cache_len = past_kvs[0][0][0].shape[1]
        # Don't exceed position embedding limit (block_size) or overshoot max tokens
        k = min(K, max_new_tokens - len(generated), block_size - cache_len - 1)
        if k <= 0:
            break
        # Draft
        candidates, draft_probs = draft_tokens(draft_model, current_token, k)
        # Verify
        target_probs, new_kvs = verify_candidates(target_model, current_token, candidates, past_kvs)
        total_target_calls += 1
        # Accept/reject
        accepted = accept_reject(candidates, draft_probs, target_probs)
        # Trim cache: keep prompt + current_token + accepted (minus the last, which is next input)
        num_to_keep = cache_len + len(accepted)
        past_kvs = trim_kv_cache(new_kvs, num_to_keep)
        generated.extend(accepted)
        current_token = accepted[-1]
    return generated[:max_new_tokens], total_target_calls
    
@torch.no_grad()
def standard_generate(target_model, prompt_tokens, max_new_tokens):
    """Standard autoregressive generation with KV cache (1 token per forward pass)."""
    target_model.eval()
    generated = []
    # Prefill
    input_ids = torch.tensor([prompt_tokens], dtype=torch.long, device=device)
    positions = torch.arange(len(prompt_tokens), device=device).unsqueeze(0)
    logits, _, past_kvs = target_model(input_ids, pos=positions)
    probs = F.softmax(logits[0, -1, :], dim=-1)
    current_token = torch.multinomial(probs, num_samples=1).item()
    generated.append(current_token)
    total_target_calls = 1
    while len(generated) < max_new_tokens:
        cache_len = past_kvs[0][0][0].shape[1]
        if cache_len >= block_size:
            break
        input_ids = torch.tensor([[current_token]], dtype=torch.long, device=device)
        positions = torch.tensor([[cache_len]], device=device)
        logits, _, past_kvs = target_model(input_ids, pos=positions, past_kvs=past_kvs)
        total_target_calls += 1
        probs = F.softmax(logits[0, -1, :], dim=-1)
        current_token = torch.multinomial(probs, num_samples=1).item()
        generated.append(current_token)
    return generated[:max_new_tokens], total_target_calls

In [24]:
# ─── BENCHMARK ───
prompt = encode("ROMEO:")
max_new = 20
# Standard
torch.manual_seed(42)
std_tokens, std_calls = standard_generate(model, prompt, max_new)
print(f"Standard:    {std_calls} target calls → {decode(std_tokens)!r}")
# Speculative
torch.manual_seed(42)  
spec_tokens, spec_calls = speculative_generate(model, draft_model, prompt, max_new, K=4)
print(f"Speculative: {spec_calls} target calls → {decode(spec_tokens)!r}")
print(f"\nTarget call reduction: {std_calls} → {spec_calls} ({std_calls/spec_calls:.2f}x)")

Standard:    20 target calls → '\nFar a whith tiden m'
Speculative: 9 target calls → "\n'But meand he count"

Target call reduction: 20 → 9 (2.22x)


### Test 1: Output equivalence (greedy)

With greedy decoding (argmax instead of sampling), speculative decoding should produce **exactly the same output** as standard decoding. This validates correctness.

In [25]:
# ─── TESTS ───
# Test 1: Greedy equivalence
@torch.no_grad()
def greedy_generate(target_model, prompt_tokens, max_new_tokens):
    """Greedy (argmax) generation for deterministic comparison."""
    target_model.eval()
    generated = []
    input_ids = torch.tensor([prompt_tokens], dtype=torch.long, device=device)
    positions = torch.arange(len(prompt_tokens), device=device).unsqueeze(0)
    logits, _, past_kvs = target_model(input_ids, pos=positions)
    current_token = logits[0, -1, :].argmax().item()
    generated.append(current_token)
    while len(generated) < max_new_tokens:
        cache_len = past_kvs[0][0][0].shape[1]
        if cache_len >= block_size:
            break
        input_ids = torch.tensor([[current_token]], dtype=torch.long, device=device)
        positions = torch.tensor([[cache_len]], device=device)
        logits, _, past_kvs = target_model(input_ids, pos=positions, past_kvs=past_kvs)
        current_token = logits[0, -1, :].argmax().item()
        generated.append(current_token)
    return generated[:max_new_tokens]


@torch.no_grad()
def speculative_generate_greedy(target_model, draft_model, prompt_tokens, max_new_tokens, K=4):
    """Greedy speculative decoding — accept iff draft == target's argmax."""
    target_model.eval()
    generated = []
    input_ids = torch.tensor([prompt_tokens], dtype=torch.long, device=device)
    positions = torch.arange(len(prompt_tokens), device=device).unsqueeze(0)
    logits, _, past_kvs = target_model(input_ids, pos=positions)
    current_token = logits[0, -1, :].argmax().item()
    generated.append(current_token)
    
    while len(generated) < max_new_tokens:
        cache_len = past_kvs[0][0][0].shape[1]
        k = min(K, max_new_tokens - len(generated), block_size - cache_len - 1)
        if k <= 0:
            break
        candidates, _ = draft_tokens(draft_model, current_token, k)
        target_probs, new_kvs = verify_candidates(target_model, current_token, candidates, past_kvs)
        accepted = []
        for i in range(len(candidates)):
            target_choice = target_probs[i].argmax().item()
            if candidates[i] == target_choice:
                accepted.append(candidates[i])
            else:
                accepted.append(target_choice)
                break
        else:
            bonus = target_probs[len(candidates)].argmax().item()
            accepted.append(bonus)
        num_to_keep = cache_len + len(accepted)
        past_kvs = trim_kv_cache(new_kvs, num_to_keep)
        generated.extend(accepted)
        current_token = accepted[-1]
    return generated[:max_new_tokens]

# Run tests
prompt = encode("First")
max_new = 20
std_out = greedy_generate(model, prompt, max_new)
spec_out = speculative_generate_greedy(model, draft_model, prompt, max_new, K=4)

print(f"Standard greedy:    {decode(std_out)!r}")
print(f"Speculative greedy: {decode(spec_out)!r}")
assert std_out == spec_out, f"MISMATCH!\n  std:  {std_out}\n  spec: {spec_out}"
print("✅ Test 1: Greedy equivalence PASSED")

Standard greedy:    ' the shall be the sh'
Speculative greedy: ' the shall be the sh'
✅ Test 1: Greedy equivalence PASSED


In [26]:
# ─── Test 2: Self-draft (target as its own draft) ───
# When p == q for every token, acceptance probability = min(1, p/q) = 1.
# Every candidate should be accepted → we always get K+1 tokens per step.
@torch.no_grad()

def self_draft_generate(target_model, prompt_tokens, max_new_tokens, K=4):
    """Use target model's own distribution as the draft. Should accept everything."""
    target_model.eval()
    generated = []
    total_accepted = 0
    total_drafted = 0
    input_ids = torch.tensor([prompt_tokens], dtype=torch.long, device=device)
    positions = torch.arange(len(prompt_tokens), device=device).unsqueeze(0)
    logits, _, past_kvs = target_model(input_ids, pos=positions)
    probs = F.softmax(logits[0, -1, :], dim=-1)
    current_token = torch.multinomial(probs, num_samples=1).item()
    generated.append(current_token)
    while len(generated) < max_new_tokens:
        cache_len = past_kvs[0][0][0].shape[1]
        k = min(K, max_new_tokens - len(generated), block_size - cache_len - 1)
        if k <= 0:
            break
        # Use target model to get draft probs (run a separate forward pass)
        # This simulates p == q exactly
        draft_input = torch.tensor([[current_token]], dtype=torch.long, device=device)
        draft_pos = torch.tensor([[cache_len]], device=device)
        draft_logits, _, _ = target_model(draft_input, pos=draft_pos, past_kvs=past_kvs)
        first_draft_probs = F.softmax(draft_logits[0, 0, :], dim=-1)
        # Sample K tokens autoregressively using target probs
        candidates = []
        draft_probs_list = []
        tok = current_token
        # For simplicity, use the same distribution (first position's) for all drafts
        # The real key: when we verify, target_probs[i] will match because same model
        temp_past = past_kvs
        temp_cache_len = cache_len
        for j in range(k):
            inp = torch.tensor([[tok]], dtype=torch.long, device=device)
            pos = torch.tensor([[temp_cache_len]], device=device)
            lg, _, temp_past = target_model(inp, pos=pos, past_kvs=temp_past)
            temp_cache_len += 1
            p = F.softmax(lg[0, 0, :], dim=-1)
            next_tok = torch.multinomial(p, num_samples=1).item()
            candidates.append(next_tok)
            draft_probs_list.append(p)
            tok = next_tok
        # Verify
        target_probs, new_kvs = verify_candidates(target_model, current_token, candidates, past_kvs)
        # Accept/reject — with p == q, everything should be accepted
        accepted = accept_reject(candidates, draft_probs_list, target_probs)
        total_drafted += k
        total_accepted += min(len(accepted), k)  # bonus doesn't count as "accepted draft"
        num_to_keep = cache_len + len(accepted)
        past_kvs = trim_kv_cache(new_kvs, num_to_keep)
        generated.extend(accepted)
        current_token = accepted[-1]
    acceptance_rate = total_accepted / total_drafted if total_drafted > 0 else 0
    return generated[:max_new_tokens], acceptance_rate

prompt = encode("First")
tokens, acc_rate = self_draft_generate(model, prompt, max_new_tokens=15, K=4)

print(f"Self-draft acceptance rate: {acc_rate:.1%}")
assert acc_rate == 1.0, f"Expected 100% acceptance when draft == target, got {acc_rate:.1%}"
print(f"✅ Test 2: Self-draft acceptance PASSED (rate={acc_rate:.1%})")

Self-draft acceptance rate: 100.0%
✅ Test 2: Self-draft acceptance PASSED (rate=100.0%)


In [27]:
# ─── Test 3: Uniform draft (worst case) ───
# A draft that predicts uniform distribution disagrees with the target maximally.
# Most tokens should be rejected, but output should still be valid text.
class UniformDraftModel:
    """Always predicts uniform distribution — worst possible draft."""
    def __init__(self, vocab_size, device):
        self.uniform = torch.ones(vocab_size, device=device) / vocab_size
    def get_probs(self, token_id):
        return self.uniform
    def sample(self, token_id):
        probs = self.uniform
        return torch.multinomial(probs, num_samples=1).item(), probs

uniform_draft = UniformDraftModel(vocab_size, device)
prompt = encode("First")
tokens, total_calls = speculative_generate(model, uniform_draft, prompt, max_new_tokens=15, K=4)
std_tokens, std_calls = standard_generate(model, prompt, max_new_tokens=15)

print(f"Uniform draft output: {decode(tokens)!r}")
print(f"Standard output:      {decode(std_tokens)!r}")
print(f"Uniform draft calls:  {total_calls}, Standard calls: {std_calls}")
# With uniform draft, acceptance rate is low, so spec calls ≈ std calls (no speedup)
# But the output should still be valid (not garbage)
assert len(tokens) == 15, f"Expected 15 tokens, got {len(tokens)}"
print("✅ Test 3: Uniform draft produces valid output (no crash, correct length)")

Uniform draft output: ' may dears,\nOn '
Standard output:      ' ment;\nAlcome m'
Uniform draft calls:  14, Standard calls: 15
✅ Test 3: Uniform draft produces valid output (no crash, correct length)


In [28]:
# ─── Test 4: Acceptance rate tracking ───
# Manually count accepted/drafted tokens and verify the rate makes sense.
@torch.no_grad()
def speculative_generate_with_stats(target_model, draft_model, prompt_tokens, max_new_tokens, K=4):
    """speculative_generate but also returns detailed acceptance stats."""
    target_model.eval()
    generated = []
    total_target_calls = 0
    total_drafted = 0
    total_accepted = 0
    per_step_accepted = []
    input_ids = torch.tensor([prompt_tokens], dtype=torch.long, device=device)
    positions = torch.arange(len(prompt_tokens), device=device).unsqueeze(0)
    logits, _, past_kvs = target_model(input_ids, pos=positions)
    total_target_calls += 1
    probs = F.softmax(logits[0, -1, :], dim=-1)
    current_token = torch.multinomial(probs, num_samples=1).item()
    generated.append(current_token)
    while len(generated) < max_new_tokens:
        cache_len = past_kvs[0][0][0].shape[1]
        k = min(K, max_new_tokens - len(generated), block_size - cache_len - 1)
        if k <= 0:
            break
        candidates, draft_probs = draft_tokens(draft_model, current_token, k)
        target_probs, new_kvs = verify_candidates(target_model, current_token, candidates, past_kvs)
        total_target_calls += 1
        accepted = accept_reject(candidates, draft_probs, target_probs)
        # Count: how many of the K draft tokens were accepted (bonus doesn't count)
        n_accepted_drafts = min(len(accepted) - 1, k) if len(accepted) > 0 else 0
        # If all K accepted + bonus, n_accepted_drafts = K
        # If rejected at position j, accepted has j+1 entries, n_accepted_drafts = j
        if len(accepted) == k + 1:
            n_accepted_drafts = k  # all K accepted
        else:
            n_accepted_drafts = len(accepted) - 1  # rejected, last is resampled
        total_drafted += k
        total_accepted += n_accepted_drafts
        per_step_accepted.append((k, n_accepted_drafts, len(accepted)))
        num_to_keep = cache_len + len(accepted)
        past_kvs = trim_kv_cache(new_kvs, num_to_keep)
        generated.extend(accepted)
        current_token = accepted[-1]
    return generated[:max_new_tokens], {
        "target_calls": total_target_calls,
        "total_drafted": total_drafted,
        "total_accepted": total_accepted,
        "acceptance_rate": total_accepted / total_drafted if total_drafted > 0 else 0,
        "tokens_per_call": len(generated) / total_target_calls,
        "per_step": per_step_accepted,
    }
prompt = encode("ROMEO:")
tokens, stats = speculative_generate_with_stats(model, draft_model, prompt, max_new_tokens=20, K=4)

print(f"Output: {decode(tokens)!r}")
print(f"Target forward passes: {stats['target_calls']}")
print(f"Total drafted: {stats['total_drafted']}")
print(f"Total accepted: {stats['total_accepted']}")
print(f"Acceptance rate: {stats['acceptance_rate']:.1%}")
print(f"Tokens per target call: {stats['tokens_per_call']:.2f}")
print(f"\nPer-step breakdown (drafted, accepted, total_output):")

for i, (drafted, accepted, output) in enumerate(stats['per_step']):
    print(f"  Step {i}: drafted={drafted}, accepted={accepted}/{drafted}, output={output} tokens")

assert stats['total_accepted'] <= stats['total_drafted'], "Can't accept more than drafted"
assert stats['acceptance_rate'] >= 0 and stats['acceptance_rate'] <= 1, "Rate must be [0, 1]"
assert stats['tokens_per_call'] >= 1.0, "Should get at least 1 token per call"
print("\n✅ Test 4: Acceptance rate tracking PASSED")

Output: '\nShall meanuss been,'
Target forward passes: 13
Total drafted: 43
Total accepted: 8
Acceptance rate: 18.6%
Tokens per target call: 1.62

Per-step breakdown (drafted, accepted, total_output):
  Step 0: drafted=4, accepted=0/4, output=1 tokens
  Step 1: drafted=4, accepted=0/4, output=1 tokens
  Step 2: drafted=4, accepted=0/4, output=1 tokens
  Step 3: drafted=4, accepted=0/4, output=1 tokens
  Step 4: drafted=4, accepted=0/4, output=1 tokens
  Step 5: drafted=4, accepted=0/4, output=1 tokens
  Step 6: drafted=4, accepted=2/4, output=3 tokens
  Step 7: drafted=4, accepted=1/4, output=2 tokens
  Step 8: drafted=4, accepted=2/4, output=3 tokens
  Step 9: drafted=4, accepted=2/4, output=3 tokens
  Step 10: drafted=2, accepted=0/2, output=1 tokens
  Step 11: drafted=1, accepted=1/1, output=2 tokens

✅ Test 4: Acceptance rate tracking PASSED
